# BMIN 5200 — Week 13 in-class exercise
## A blood pressure agent, and the guardrail it needs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week13.ipynb)

**Time:** ~25 minutes · **Pairs with:** Agentic AI

### What you'll do
- Build the deck's patient blood-pressure agent as an explicit perceive-decide-act loop over four registered tools
- Watch it act on a six-week-old reading, spin forever, and escalate a typo to a nurse
- Add three guardrails — a step cap, a staleness check, and a human confirmation before anything irreversible — and re-run the same three cases
- Recognize the planner for what it is: the Week 6 inference engine, wearing new clothes

### Why it matters
Almost every clinical agent demo you will be shown hides its control flow inside a model API, which makes the interesting part invisible. Here the "LLM" is a stub function you can read in twenty seconds, so the loop, the tool registry, and the failure modes are all in plain sight. The three failures below are not contrived: stale data, unbounded retries, and irreversible actions taken on bad input are what actually goes wrong when these systems reach a clinic.

Setup. Nothing to install. Note that `TODAY` is pinned rather than read from the clock, so the
staleness arithmetic gives the same answer in class today as it will in April.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(5200)
pd.set_option("display.width", 120)

TODAY = date(2026, 4, 2)          # pinned so "45 days ago" means the same thing on every laptop

## The environment: three patients on a hypertension remote-monitoring program

Per the "Environment and Tools" slide, an agent needs a world to perceive and act on. Ours is a
small in-memory registry of home blood-pressure readings and medication lists. Three patients,
each chosen to break the agent in a different way — but read them as ordinary charts for now.

In [ ]:
def home_readings(days_ago_list, systolic_center, diastolic_center):
    """Simulated home cuff readings, jittered so they look like real self-measurements."""
    return [{"measured_on": TODAY - timedelta(days=days_ago),
             "systolic": int(systolic_center + rng.normal(0, 6)),
             "diastolic": int(diastolic_center + rng.normal(0, 4))}
            for days_ago in days_ago_list]


PATIENTS = {
    "PT-1041": {  # uncontrolled, recent readings, already on two agents
        "readings": home_readings([1, 3, 5, 8], 168, 94),
        "medications": ["lisinopril 20 mg daily", "amlodipine 5 mg daily"],
    },
    "PT-2277": {  # readings look alarming, but the newest is six weeks old
        "readings": home_readings([45, 47, 52], 178, 96),
        "medications": ["hydrochlorothiazide 25 mg daily"],
    },
    "PT-3390": {  # one reading is a transcription error: 1780, not 178
        "readings": [{"measured_on": TODAY - timedelta(days=2),
                      "systolic": 1780, "diastolic": 96}],
        "medications": ["lisinopril 40 mg daily", "chlorthalidone 25 mg daily"],
    },
}

for patient_id, chart in PATIENTS.items():
    newest = max(reading["measured_on"] for reading in chart["readings"])
    print(f"{patient_id}: {len(chart['readings'])} readings, newest {newest}, "
          f"{len(chart['medications'])} antihypertensives")

## Part 1 — Tools, a planner, and the loop that joins them

Three pieces, matching the "Agent Architecture Overview" slide. **Tools** are ordinary Python
functions the agent may call. The **planner** looks at what the agent currently knows and names
the next tool. The **loop** runs planner, calls the tool, records what came back, and repeats —
perceive, decide, act.

Two of these tools change the world outside the agent, and are marked `irreversible`. Once a
nurse has been paged, no later step un-pages her.

In [ ]:
NURSE_QUEUE = []
APPOINTMENTS = []


def get_recent_readings(patient_id):
    """Perceive: pull home cuff readings from the monitoring platform."""
    return PATIENTS[patient_id]["readings"]


def check_medication_list(patient_id):
    """Perceive: what antihypertensives is this patient already on?"""
    return PATIENTS[patient_id]["medications"]


def flag_for_nurse_review(patient_id, reason):
    """Act, irreversibly: page the hypertension nurse. Cannot be undone."""
    NURSE_QUEUE.append({"patient_id": patient_id, "reason": reason})
    return f"paged hypertension nurse for {patient_id}"


def schedule_followup(patient_id, days):
    """Act, irreversibly: take a slot on a real clinic calendar."""
    APPOINTMENTS.append({"patient_id": patient_id, "on": TODAY + timedelta(days=days)})
    return f"follow-up booked for {patient_id} in {days} days"


TOOLS = {
    "get_recent_readings":  {"run": get_recent_readings,  "irreversible": False},
    "check_medication_list": {"run": check_medication_list, "irreversible": False},
    "flag_for_nurse_review": {"run": flag_for_nurse_review, "irreversible": True},
    "schedule_followup":     {"run": schedule_followup,     "irreversible": True},
}

Here is the "LLM". It is a rule base: each branch matches a condition against what the agent
currently knows and returns the tool to fire next. Compare it, line for line, with the forward
chainer you wrote in Week 6 — `state` is working memory, each `if` is a production rule, and the
first match wins, which is the simplest conflict-resolution strategy there is. Replacing this
function with a call to a frontier model changes how the next tool is chosen and changes nothing
else in the architecture.

In [ ]:
def planner(state):
    """Rule base. Returns (tool_name, arguments); ("stop", {}) when there is nothing left to do."""
    patient_id = state["patient_id"]

    if "readings" not in state:
        return "get_recent_readings", {"patient_id": patient_id}

    if state["mean_systolic"] >= 160 and "medications" not in state:
        return "check_medication_list", {"patient_id": patient_id}

    if state["mean_systolic"] >= 160 and len(state.get("medications", [])) >= 2:
        return "flag_for_nurse_review", {
            "patient_id": patient_id,
            "reason": f"mean systolic {state['mean_systolic']:.0f} on two agents"}

    if state["mean_systolic"] >= 160:
        return "schedule_followup", {"patient_id": patient_id, "days": 7}

    if state["mean_systolic"] >= 140:
        return "schedule_followup", {"patient_id": patient_id, "days": 30}

    return "stop", {}


def observe(state, tool_name, result):
    """Write the tool's result into working memory, the way a forward chainer asserts a fact."""
    if tool_name == "get_recent_readings" and len(result) > 0:
        state["readings"] = result
        state["mean_systolic"] = float(np.mean([r["systolic"] for r in result]))
    elif tool_name == "check_medication_list":
        state["medications"] = result
    elif tool_name in ("flag_for_nurse_review", "schedule_followup"):
        state["done"] = True
    return state


def run_agent(patient_id, verbose=True):
    """Perceive, decide, act — until the planner says stop."""
    state = {"patient_id": patient_id}
    step_count = 0
    while not state.get("done"):
        tool_name, arguments = planner(state)
        if tool_name == "stop":
            if verbose:
                print(f"  step {step_count}: stop")
            break
        result = TOOLS[tool_name]["run"](**arguments)
        state = observe(state, tool_name, result)
        step_count += 1
        if verbose:
            shown = result if isinstance(result, str) else f"{len(result)} items"
            print(f"  step {step_count}: {tool_name} -> {shown}")
    return state


print("PT-1041, uncontrolled on two agents:")
final_state = run_agent("PT-1041")
print(f"\nNurse queue: {NURSE_QUEUE}")

That is the "Multi-Step Execution Example" slide, running. The agent pulled readings, saw a mean
systolic of 166, checked that the patient was already on two agents, and escalated — three steps,
no model, and a defensible clinical decision. It is worth noticing how little machinery this took,
and how much of what people call "agentic" is this loop plus a bigger planner.

## Part 2 — Three ways it breaks

Now the "When Things Break" slide. Run the same agent on the other two patients. Nothing about
the code changes; only the data does.

In [ ]:
print("PT-2277 — readings are alarming, and six weeks old:")
run_agent("PT-2277")
newest = max(r["measured_on"] for r in PATIENTS["PT-2277"]["readings"])
print(f"  (newest reading was taken {(TODAY - newest).days} days ago)\n")

print("PT-3390 — one reading, and it is a transcription error:")
run_agent("PT-3390")
print(f"  (the systolic the agent acted on: "
      f"{PATIENTS['PT-3390']['readings'][0]['systolic']} mmHg)\n")

print("Nurse queue after both runs:")
print(pd.DataFrame(NURSE_QUEUE).to_string(index=False))
print("\nAppointments booked:")
print(pd.DataFrame(APPOINTMENTS).to_string(index=False))

Two harms, both irreversible. PT-2277 got a clinic slot booked off a six-week-old cuff reading
that says nothing about today, and the signal that actually mattered — this patient stopped
measuring five weeks ago — went unnoticed. PT-3390 got the hypertension nurse paged on a systolic
of 1780 mmHg, a number no living person has produced, because nothing in the loop asks whether an
input is physiologically possible before acting on it.

The third failure needs its own cell, because it does not terminate.

In [ ]:
# A patient enrolled in the program who has not submitted a single reading.
PATIENTS["PT-4008"] = {"readings": [], "medications": []}

state = {"patient_id": "PT-4008"}
trace = []
for emergency_stop in range(20):        # this range() is the notebook's seatbelt, not the agent's
    tool_name, arguments = planner(state)
    if tool_name == "stop":
        break
    result = TOOLS[tool_name]["run"](**arguments)
    state = observe(state, tool_name, result)
    trace.append(tool_name)

print(f"First 8 steps: {trace[:8]}")
print(f"Total steps before the notebook cut it off: {len(trace)}")
print("\nThe planner asks for readings, `observe` refuses to record an empty list, so the")
print("planner asks for readings again. Nothing in the agent stops this. Left alone it would")
print("call the monitoring API until someone noticed the bill.")

## Predict before you run

You are about to add three guardrails, matching the "Safety Layers" slide: a **step cap**, a
**staleness check**, and a **human confirmation** before any tool marked irreversible.

**Commit to answers as a room before you write any code.**
1. Which of the three guardrails fixes PT-2277, the six-week-old readings?
2. Which one fixes PT-3390, the 1780 mmHg typo — and is the answer the same as your instinct,
   which was probably "validate the input"? What is the difference between those two fixes?
3. PT-1041 was handled correctly. Will any guardrail change what happens to that patient?

## Part 3 — Three guardrails, three TODOs

Each guardrail is one small function, and each currently returns `True`, which is another way of
saying the guardrail is switched off. The loop below is already wired to call all three; you only
have to make them say no in the right circumstances.

In [ ]:
MAX_STEPS = 6
FRESH_ENOUGH_DAYS = 14
PLAUSIBLE_SYSTOLIC = (60, 260)


def within_step_budget(step_count):
    """Guardrail 1: an agent that cannot stop itself is not safe to run unattended."""
    # TODO: return False once step_count reaches MAX_STEPS.
    # The 20 below is only here so the notebook terminates; the agent still has no real cap.
    return step_count < 20


def readings_are_fresh(readings):
    """Guardrail 2: old data is not the same as current data, and the agent cannot tell."""
    # TODO: return False when the newest reading is more than FRESH_ENOUGH_DAYS old.
    return True


def human_approves(tool_name, arguments, state):
    """Guardrail 3: a standing clinician, simulated. Refuse implausible physiology."""
    # TODO: return False if any recorded systolic falls outside PLAUSIBLE_SYSTOLIC.
    return True


def run_guarded_agent(patient_id):
    """The same loop as before, with the three safety layers wired in."""
    state = {"patient_id": patient_id}
    step_count = 0
    while not state.get("done"):
        if not within_step_budget(step_count):
            print(f"  HALTED: step budget exhausted after {step_count} steps; "
                  f"handing {patient_id} to a human")
            break

        tool_name, arguments = planner(state)
        if tool_name == "stop":
            print("  step: stop")
            break

        if tool_name == "get_recent_readings":
            readings = TOOLS[tool_name]["run"](**arguments)
            if len(readings) > 0 and not readings_are_fresh(readings):
                age = (TODAY - max(r["measured_on"] for r in readings)).days
                print(f"  BLOCKED: newest reading is {age} days old; requesting a fresh "
                      f"measurement instead of acting")
                break
            state = observe(state, tool_name, readings)
            step_count += 1
            print(f"  step {step_count}: {tool_name} -> {len(readings)} items")
            continue

        if TOOLS[tool_name]["irreversible"] and not human_approves(tool_name, arguments, state):
            print(f"  BLOCKED: clinician declined {tool_name} for {patient_id}")
            break

        result = TOOLS[tool_name]["run"](**arguments)
        state = observe(state, tool_name, result)
        step_count += 1
        print(f"  step {step_count}: {tool_name} -> {result}")
    return state


NURSE_QUEUE.clear()
APPOINTMENTS.clear()
for patient_id in ["PT-1041", "PT-2277", "PT-3390", "PT-4008"]:
    print(f"{patient_id}:")
    run_guarded_agent(patient_id)
    print()

print(f"Irreversible actions taken this run: "
      f"{len(NURSE_QUEUE)} pages, {len(APPOINTMENTS)} appointments")

With all three TODOs filled in, PT-1041 is still handled exactly as before — a guardrail that
changes correct behaviour is a bad guardrail — while PT-2277 is blocked as stale, PT-3390 is
blocked by the clinician, and PT-4008 halts at six steps and gets handed to a person. One page,
zero wasted appointments.

Notice what the second guardrail does *not* do. It does not correct the stale reading; it refuses
to act and escalates to a human. Most real safety layers are like this: they convert a silent
wrong action into a loud non-action, which is worth a great deal in a clinic and is not at all
the same thing as making the agent smarter.

## Part 4 — Closing the loop

Two things to say out loud before the semester ends.

**The planner is the Week 6 inference engine.** Working memory, production rules, first-match
conflict resolution, forward chaining until quiescence. We changed the vocabulary — "tools"
instead of "actions", "agent" instead of "expert system" — and we let the rules call out to the
world instead of only asserting facts. The control structure is the one Shortliffe's group was
running on MYCIN in 1976. When you read that an agent "reasons and plans", ask which of those two
things is new and which is a rename.

**And back to Week 1.** The first notebook of the semester ended with ELIZA, twenty lines of
regular expressions that people confided in, and a question you had to answer in writing on day
one: *what would a system have to be able to do before you would say it genuinely understands a
patient saying "my chest hurts when I climb stairs"?* You filled in
`understanding_requirements`. Open Week 1, find what you wrote, and put it back below.

In [ ]:
understanding_requirements = """
TODO: paste what you wrote in Week 1, then add a line saying what you would change now.
"""

print(understanding_requirements)
print("Since Week 1 you have built, by hand:")
for week, capability in [
    (2, "a propositional model checker, and the point where truth tables stop scaling"),
    (3, "a walk over a real biomedical ontology"),
    (4, "BFS, DFS, greedy and A*, with the node counts to compare them"),
    (5, "a genetic algorithm and a particle swarm"),
    (6, "a forward and backward chaining engine, and a why() facility"),
    (7, "the same rule base again, in CLIPS"),
    (8, "a Bayesian network you could interrogate with evidence"),
    (9, "mutual information, and a decision tree induced from data"),
    (10, "a tokenizer, attention from the formula, and contextual embeddings"),
    (11, "post-hoc explanations of a model nobody can read"),
    (12, "a fairness audit that found a flaw planted three weeks earlier"),
    (13, "an agent loop, and the guardrails it turned out to need"),
]:
    print(f"  Week {week:2d}: {capability}")
print("\nNone of it understands a patient complaint. Which of these get you closer, and")
print("which only make the system act more as though it does?")

## Talk about it

1. The staleness guardrail refuses to act on data older than 14 days. Where did 14 come from? Who
   at a health system should own that number, and what happens the first time a clinician
   overrides it?
2. Guardrail 3 puts a human in front of every irreversible action. Run that forward: the nurse
   approves 200 of these a week, 198 of them obviously fine. What is the failure mode of
   human-in-the-loop review at that volume, and does the guardrail still do anything?
3. Swap the stub planner for a real language model and the architecture is unchanged, but the
   planner's decisions are no longer auditable line by line. Which of today's three guardrails
   would you trust more in that setting, and which would you trust less?

## Solutions

Completed versions of the three TODOs, as markdown so they do not run.

```python
def within_step_budget(step_count):
    return step_count < MAX_STEPS


def readings_are_fresh(readings):
    newest = max(reading["measured_on"] for reading in readings)
    return (TODAY - newest).days <= FRESH_ENOUGH_DAYS


def human_approves(tool_name, arguments, state):
    low, high = PLAUSIBLE_SYSTOLIC
    for reading in state.get("readings", []):
        if not (low <= reading["systolic"] <= high):
            print(f"    clinician sees systolic {reading['systolic']} mmHg and refuses")
            return False
    return True
```

Which produces:

```
PT-1041:
  step 1: get_recent_readings -> 4 items
  step 2: check_medication_list -> ['lisinopril 20 mg daily', 'amlodipine 5 mg daily']
  step 3: flag_for_nurse_review -> paged hypertension nurse for PT-1041

PT-2277:
  BLOCKED: newest reading is 45 days old; requesting a fresh measurement instead of acting

PT-3390:
  step 1: get_recent_readings -> 1 items
  step 2: check_medication_list -> ['lisinopril 40 mg daily', 'chlorthalidone 25 mg daily']
    clinician sees systolic 1780 mmHg and refuses
  BLOCKED: clinician declined flag_for_nurse_review for PT-3390

PT-4008:
  step 1: get_recent_readings -> 0 items
  ...
  step 6: get_recent_readings -> 0 items
  HALTED: step budget exhausted after 6 steps; handing PT-4008 to a human

Irreversible actions taken this run: 1 pages, 0 appointments
```